# Import required packages

In [25]:
from datasets import load_dataset
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer, AutoModelForCausalLM, AutoTokenizer
from torch.nn.functional import normalize
from typing import List
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import get_scheduler
from sentence_transformers import SentenceTransformer, util
from difflib import SequenceMatcher

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
KEEP_RATIO = 0.50
MAX_NEW_TOKENS = 64
DEVICE = torch.device("cpu")
MAX_LEN = 128
tokenizer = LlamaTokenizer.from_pretrained(MODEL_NAME)

# Load LLM Lingua Dataset from HuggingFace

In [29]:
data = load_dataset("microsoft/MeetingBank-LLMCompressed", split="train")
print(len(data))
for idx, sample in enumerate(data):
    # concatenation of all chunks
    prompt = sample["prompt"]
    compressed_prompt = sample["compressed_prompt"]

5169


# Line up original tokens with compressed ones. Teaches model which tokens should be kept

In [ ]:
def align_tokens(prompt: str, compressed: str):
    orig_tokens = tokenizer.tokenize(prompt)[:128]
    comp_tokens = tokenizer.tokenize(compressed)[:128]

    matcher = SequenceMatcher(None, orig_tokens, comp_tokens)

    labels = []
    for tag, i1, i2, _, _ in matcher.get_opcodes():
        if tag == 'equal':
            labels.extend([1] * (i2 - i1))
        else:
            labels.extend([0] * (i2 - i1))

    input_ids = tokenizer.convert_tokens_to_ids(orig_tokens)
    assert len(input_ids) == len(labels)
    return input_ids, labels

# Define classes for compression dataset and scorer

In [ ]:
class TokenCompressionDataset(Dataset):
    def __init__(self, dataset, tokenizer):
        self.items = []
        for sample in dataset:
            try:
                ids, lbls = align_tokens(sample["prompt"], sample["compressed_prompt"])
                if ids:
                    self.items.append((ids, lbls))
            except:
                continue
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        ids, lbls = self.items[idx]
        enc = self.tokenizer.prepare_for_model(
            ids,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = torch.tensor(
            lbls[:MAX_LEN] + [0] * max(0, MAX_LEN - len(lbls)),
            dtype=torch.float
        )
        return input_ids, attention_mask, labels

In [ ]:
class CompressionScorer(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)

        # Use one TransformerEncoder layer
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=4,
            dim_feedforward=hidden_size * 4,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=1)

        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None):
        x = self.embed(input_ids)
        if attention_mask is not None:
            # Ignore padding
            x = self.encoder(
                x,
                src_key_padding_mask=~attention_mask.bool()
            )
        else:
            x = self.encoder(x)

        logits = self.classifier(x).squeeze(-1)
        return logits

# Train model on compression dataset

In [11]:
dataset = TokenCompressionDataset(data, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = CompressionScorer(vocab_size=tokenizer.vocab_size).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
scheduler = get_scheduler("linear", opt, num_warmup_steps=100, num_training_steps=len(loader)*3)
loss_fn = nn.BCEWithLogitsLoss()

model.train()
for epoch in range(3):
    for batch in loader:
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels.float())

        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()

    print(f"Epoch {epoch+1}: loss={loss.item():.4f}")


c:\Users\ammil\Documents\College\ACM-Research-F25\.venv\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1: loss=0.5955
Epoch 2: loss=0.6281
Epoch 3: loss=0.6336


# Define functions to score and compress prompts based on trained model + dataset

In [12]:
def score_tokens(scorer_model, input_ids, attention_mask):
    scorer_model.eval()
    with torch.no_grad():
        logits = scorer_model(input_ids.to(DEVICE), attention_mask.to(DEVICE))
        probs = torch.sigmoid(logits).squeeze(0)  # shape: (seq_len,)
    return probs

In [13]:
def compress_prompt(input_ids, scores, keep_ratio):
    k = max(1, int(len(scores) * keep_ratio))
    top_idx = torch.topk(scores, k).indices.sort()[0]
    return input_ids[:, top_idx]

In [ ]:
def generate_with_compression(
    model, tokenizer, prompt: str, keep_ratio: float
) -> str:
    # tokenize
    enc = tokenizer(prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LEN)
    input_ids, attn = enc["input_ids"], enc["attention_mask"]

    # importance score
    scores = score_tokens(model, input_ids, attn)
    compressed_ids = compress_prompt(input_ids, scores, keep_ratio)
    out = model.generate(
        input_ids=compressed_ids.to(DEVICE),
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test compression and give similarity score

In [24]:
stransform = SentenceTransformer("all-MiniLM-L6-v2")

def test_prompts_similarity(prompts, model, tokenizer, keep_ratio=0.5):
    similarities = []

    for text in prompts:
        enc = tokenizer(text, return_tensors="pt")
        input_ids, attn = enc["input_ids"], enc["attention_mask"]

        scores = score_tokens(model, input_ids, attn)
        compressed_ids = compress_prompt(input_ids, scores, keep_ratio=keep_ratio)

        compressed = tokenizer.decode(compressed_ids[0], skip_special_tokens=True)

        emb1 = stransform.encode(text, convert_to_tensor=True)
        emb2 = stransform.encode(compressed, convert_to_tensor=True)

        sim = util.cos_sim(emb1, emb2).item()
        similarities.append(sim)

        print(f"\nOriginal:   {text}")
        print(f"Compressed: {compressed}")
        print(f"Similarity: {sim:.4f}")

    avg_sim = sum(similarities) / len(similarities)
    print(f"\nAverage similarity over {len(prompts)} prompts: {avg_sim:.4f}")
    return avg_sim


# Example usage:
test_prompts = [
    "Explain the fundamental theorem of calculus to a high school student.",
    "Summarize the key points of the French Revolution.",
    "What are the health benefits of intermittent fasting?",
    "How does a blockchain maintain security without a central authority?",
    "Describe the process of photosynthesis in simple terms.",
]

test_prompts_similarity(test_prompts, model, tokenizer, keep_ratio=0.5)



Original:   Explain the fundamental theorem of calculus to a high school student.
Compressed: Explain the theorem of a student
Similarity: 0.6462

Original:   Summarize the key points of the French Revolution.
Compressed: ize the key the French Revolution
Similarity: 0.7308

Original:   What are the health benefits of intermittent fasting?
Compressed: What the health intermitt fast
Similarity: 0.6165

Original:   How does a blockchain maintain security without a central authority?
Compressed: How blockchain maintain security without
Similarity: 0.7632

Original:   Describe the process of photosynthesis in simple terms.
Compressed: Describe thethesis simple terms
Similarity: 0.6301

Average similarity over 5 prompts: 0.6774


0.6773598551750183